# 🏦 Loan Underwriting & Risk Assessment — Training Notebook

This notebook contains the complete pipeline for fine-tuning a Llama-3-8B model to perform autonomous loan underwriting using **Unsloth** and **TRL**.

Built for the **Scaler x Meta PyTorch Hackathon (OpenEnv Round 1)**.

## 1. Installation
We use Unsloth for 2x faster training and 70% less memory usage.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

## 2. Configuration & Model Loading

In [ ]:
import torch
from unsloth import FastLanguageModel

model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"
max_seq_length = 2048
dtype = None # None for auto detection
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

## 3. Data Preparation

In [ ]:
from datasets import load_dataset
import math

def get_training_data():
    raw = load_dataset("AnguloM/loan_data", split="train")
    formatted = []
    for row in raw:
        income = round(math.exp(row["log.annual.inc"]), 2)
        dti = row["dti"]
        fico = row["fico"]
        purpose = row["purpose"].replace("_", " ")
        
        # Simple heuristic labeling for SFT
        if fico < 650 or dti > 40:
            risk, decision, rate = "High", "Reject", "14%+"
        elif fico >= 720 and dti < 20:
            risk, decision, rate = "Low", "Approve", "7-9%"
        else:
            risk, decision, rate = "Medium", "Conditional Approve", "10-13%"
            
        profile = f"Income: ${income}, FICO: {fico}, DTI: {dti}%, Purpose: {purpose}"
        text = f"### Instruction:\nEvaluate this loan application: {profile}\n\n### Response:\n" \
               f'{{"risk_level":"{risk}","decision":"{decision}","interest_rate":"{rate}"}}'
        formatted.append({"text": text + tokenizer.eos_token})
    return formatted

dataset = get_training_data()

## 4. Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = Dataset.from_list(dataset),
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

## 5. Saving and Deployment

In [ ]:
model.save_pretrained("loan_underwriting_lora")
tokenizer.save_pretrained("loan_underwriting_lora")

# To push to Hub:
# model.push_to_hub("your-username/loan-underwriting-lora", token="your-token")